# Vapor Policy Impact — Databricks Walkthrough

End-to-end walkthrough of the `vapor_policy_impact` causal inference + counterfactual
forecasting framework, for estimating the impact of a state-level vapor policy on
tobacco retail volume *before* it is implemented in a new state.

See `Vapor-Policy-Impact-Forecasting-Methodology.md` for the full methodology write-up
this notebook implements, and `README.md` for package-level docs.

**Synthetic vs. real data.** By default this notebook runs against a synthetic,
simulated panel with a known, injected policy effect (no real retail-scan data was
available when this framework was built, so every component was validated against
ground truth first). Flip `USE_SYNTHETIC_DATA = False` in the **Configuration** cell
below and fill in your own data source to run against real data — everything after
that cell is unchanged either way; nothing downstream cares where the panel came from.

**Read this before trusting a number.** Every simplification relative to the full
methodology (the event-study estimator is a from-scratch simplified
Callaway–Sant'Anna, the "BSTS" baseline is a frequentist state-space fit via
`statsmodels`, etc.) is documented in the relevant module's docstring in
`vapor_policy_impact/`. Leave-one-state-out validation during development showed
~100% sign accuracy but weak rank-correlation on effect *magnitude* with only 9
historical treated states — expect similar caveats on your own data, and lean on the
LOSO/placebo sections below rather than trusting a single point estimate.


## 0. Environment setup

Installs the framework's dependencies on the cluster. Safe to re-run / run locally too
(a no-op if everything's already installed). On Databricks this will prompt a Python
interpreter restart the first time — that's expected.


In [ ]:
%pip install -q "numpy>=1.26" "pandas>=2.1" "scipy>=1.11" "statsmodels>=0.14" "scikit-learn>=1.3" "matplotlib>=3.8"

In [ ]:
try:
    dbutils.library.restartPython()  # noqa: F821 -- only defined on Databricks
except NameError:
    pass

## 0b. Locate and import the `vapor_policy_impact` package

Works whether this notebook was opened via **Databricks Repos** (clone this branch as
a Repo — the package sits right next to this notebook) or a local Jupyter checkout. If
auto-discovery fails for your setup, set `REPO_ROOT_OVERRIDE` below explicitly.


In [ ]:
import os
import sys


def _find_repo_root(start: str, marker: str = "vapor_policy_impact", max_up: int = 6) -> str:
    d = os.path.abspath(start)
    for _ in range(max_up):
        if os.path.isdir(os.path.join(d, marker)):
            return d
        parent = os.path.dirname(d)
        if parent == d:
            break
        d = parent
    return start


REPO_ROOT_OVERRIDE = None  # e.g. "/Workspace/Repos/you@company.com/Data-Science--Cheat-Sheet"
REPO_ROOT = REPO_ROOT_OVERRIDE or _find_repo_root(os.getcwd())
if REPO_ROOT not in sys.path:
    sys.path.append(REPO_ROOT)

print(f"Using REPO_ROOT = {REPO_ROOT}")
assert os.path.isdir(os.path.join(REPO_ROOT, "vapor_policy_impact")), (
    "Could not find the vapor_policy_impact package automatically. Set REPO_ROOT_OVERRIDE "
    "above to the checked-out repo path (e.g. your Databricks Repos folder for this branch)."
)

In [ ]:
# Databricks defines `display()` globally with a rich native table/plot UI; fall back to
# IPython's for local Jupyter so every display(...) call below works in both places.
try:
    display  # noqa: F821
except NameError:
    from IPython.display import display

## 0c. Framework imports

In [ ]:
import numpy as np
import pandas as pd

from vapor_policy_impact.config import CATEGORIES
from vapor_policy_impact.data.simulate import simulate_panel
from vapor_policy_impact.data.loaders import (
    date_to_week_index,
    load_policy_calendar,
    load_real_panel,
    load_state_covariates,
)
from vapor_policy_impact.pipeline import prepare_covariates, run_category_pipeline
from vapor_policy_impact.decomposition.manufacturer import decompose_manufacturer
from vapor_policy_impact.substitution.cross_category import category_residual_correlation, reconcile_categories
from vapor_policy_impact.features.engineering import build_aggregated_feature_series
from vapor_policy_impact.reporting.business_output import build_executive_table
from vapor_policy_impact.reporting.visuals import (
    plot_event_study,
    plot_scenario_fan_chart,
    plot_share_shift,
    plot_substitution_waterfall,
)
from vapor_policy_impact.validation.loso import (
    loso_aggregate_metrics,
    run_loso,
    run_placebo_in_time,
    summarize_loso,
)

## 1. Configuration — point this at your data

This is the only cell you should need to edit to run against your own data. Leave
`USE_SYNTHETIC_DATA = True` to run the demo against the synthetic simulator with a
known, injected effect (no real data needed, and a good way to sanity-check the
notebook runs end-to-end in your environment before pointing it at anything real).


In [ ]:
# ============================================================================
# CONFIGURATION
# ============================================================================

USE_SYNTHETIC_DATA = True

# --- only used when USE_SYNTHETIC_DATA = False --------------------------------

# A Delta table, Parquet, or CSV path readable by Spark: a DBFS path, a Unity Catalog
# Volume path, or `catalog.schema.table` (read via spark.table() -- see the loading
# cell below).
DATA_SOURCE_PATH = "/Volumes/catalog/schema/volume/retail_scan_panel"
DATA_SOURCE_FORMAT = "delta"  # "delta" | "parquet" | "csv" | "table"

# our_column_name -> your_column_name. Only state/week/category/manufacturer_group/volume
# are required -- everything else is optional and features degrade gracefully without it
# (see vapor_policy_impact/features/engineering.py).
COLUMN_MAPPING = {
    "state": "STATE_CD",
    "week": "WEEK_END_DATE",        # a real calendar date column, converted automatically below
    "category": "CATEGORY_DESC",
    "manufacturer_group": "MFG_GROUP_DESC",
    "volume": "UNIT_VOLUME",
    # "manufacturer": "MANUFACTURER_NM",
    # "sku": "SKU_ID",
    # "brand": "BRAND_NM",
    # "sales": "DOLLAR_SALES",
    # "price": "AVG_UNIT_PRICE",
    # "promo_depth": "PROMO_PCT_ACV",
    # "distribution_acv": "TOTAL_PCT_ACV",
}
WEEK_IS_DATE_COLUMN = True  # False if your `week` column is already an integer week index

# state -> policy effective date (or integer week index if WEEK_IS_DATE_COLUMN=False).
# Only include states that have ALREADY implemented the policy -- every other state in
# your data becomes a potential control / synthetic-control donor-pool state.
POLICY_CALENDAR = {
    # "Ohio": "2024-03-01",
    # "Colorado": "2023-11-15",
}

# Optional: a small state-level demographics table (population, urbanization, income,
# border_state flag, retail_density -- see pipeline.DEFAULT_COVARIATE_COLS). Leave as
# None to fall back to a minimal covariate set derived from the panel itself, which
# degrades Layer 3's ability to personalize the transported effect to your target
# state (see vapor_policy_impact/data/loaders.py::build_fallback_state_covariates).
STATE_COVARIATES_PATH = None
STATE_COVARIATES_FORMAT = "delta"
STATE_COVARIATES_COLUMN_MAPPING = None  # e.g. {"state": "STATE_CD", "population_m": "POP_MILLIONS", ...}

# The state you want the 13-week Policy vs. No-Policy forecast for, and the date its
# policy would take (or took) effect.
TARGET_STATE = "New State"
TARGET_STATE_EFFECTIVE_DATE = "2026-06-01"

# Which manufacturer_group labels in YOUR data correspond to Altria vs. competitors
# (used by the decomposition section -- only relevant if manufacturer_group is populated).
ALTRIA_LABEL = "Altria"
COMPETITOR_LABEL = "Competitor"

# Override the donor pool explicitly, or leave None to default to every state in the
# panel that is neither the target state nor already in POLICY_CALENDAR.
DONOR_POOL_STATES_OVERRIDE = None

# How much pre-period history (in weeks) the synthetic-control/baseline forecaster looks
# back over. The framework default is 120; lower this if your real history is shorter.
PRE_PERIOD_LOOKBACK_WEEKS = 120

# LOSO validation takes ~10-15 minutes even at reduced settings (it refits the full
# stack once per historical treated state) -- off by default, flip on deliberately.
RUN_LOSO_VALIDATION = False

# Bootstrap / Monte Carlo sizing -- lower for faster, noisier iteration; raise for a
# final run. See examples/run_demo.py for the defaults used there.
N_BOOTSTRAP_MAIN = 150
N_BOOTSTRAP_VALIDATION = 60
N_MC = 3000

In [ ]:
# A few of the above are also exposed as Databricks widgets for convenience when running
# this notebook as a parameterized job. No-ops outside Databricks.
try:
    dbutils.widgets.dropdown("use_synthetic_data", str(USE_SYNTHETIC_DATA), ["True", "False"])
    dbutils.widgets.text("data_source_path", DATA_SOURCE_PATH)
    dbutils.widgets.text("target_state", TARGET_STATE)
    USE_SYNTHETIC_DATA = dbutils.widgets.get("use_synthetic_data") == "True"
    DATA_SOURCE_PATH = dbutils.widgets.get("data_source_path")
    TARGET_STATE = dbutils.widgets.get("target_state")
except NameError:
    pass

## 2. Load data

In [ ]:
if USE_SYNTHETIC_DATA:
    sim = simulate_panel()
    panel = sim.panel
    policy_calendar = sim.policy_calendar
    state_covariates_raw = sim.state_covariates
    target_state = sim.target_state
    target_effective_week = sim.target_state_effective_week
    donor_pool_states = sim.donor_pool_states
    altria_label, competitor_label = "Altria", "Competitor"
    true_no_policy_for_loso = sim.full_ground_truth_no_policy  # ground truth only exists for synthetic data
    print(f"Synthetic panel: {len(panel):,} rows | target state: {target_state} | "
          f"target policy week: {target_effective_week}")

else:
    # `spark` is provided automatically in a Databricks notebook. Falls back to a local
    # pandas read so the real-data branch can still be smoke-tested outside Databricks.
    if DATA_SOURCE_FORMAT == "table":
        raw = spark.table(DATA_SOURCE_PATH)
    elif DATA_SOURCE_FORMAT == "csv":
        try:
            raw = spark.read.option("header", True).option("inferSchema", True).csv(DATA_SOURCE_PATH)
        except NameError:
            raw = pd.read_csv(DATA_SOURCE_PATH)
    else:
        try:
            raw = spark.read.format(DATA_SOURCE_FORMAT).load(DATA_SOURCE_PATH)
        except NameError:
            raw = pd.read_parquet(DATA_SOURCE_PATH)

    panel, week_zero_date = load_real_panel(raw, COLUMN_MAPPING, week_is_date=WEEK_IS_DATE_COLUMN)
    policy_calendar = load_policy_calendar(POLICY_CALENDAR, week_zero_date=week_zero_date)

    if STATE_COVARIATES_PATH:
        try:
            cov_source = spark.read.format(STATE_COVARIATES_FORMAT).load(STATE_COVARIATES_PATH)
        except NameError:
            cov_source = pd.read_parquet(STATE_COVARIATES_PATH)
    else:
        cov_source = None
    state_covariates_raw = load_state_covariates(
        cov_source, panel["state"].unique(),
        column_mapping=STATE_COVARIATES_COLUMN_MAPPING, panel_for_fallback=panel,
    )

    target_state = TARGET_STATE
    target_effective_week = (
        date_to_week_index(TARGET_STATE_EFFECTIVE_DATE, week_zero_date)
        if WEEK_IS_DATE_COLUMN else int(TARGET_STATE_EFFECTIVE_DATE)
    )
    donor_pool_states = DONOR_POOL_STATES_OVERRIDE or [
        s for s in panel["state"].unique() if s != target_state and s not in policy_calendar.treated_states
    ]
    altria_label, competitor_label = ALTRIA_LABEL, COMPETITOR_LABEL
    true_no_policy_for_loso = None  # unobservable for real data -- LOSO falls back to actuals-only scoring

    print(f"Real panel: {len(panel):,} rows | states: {panel['state'].nunique()} | "
          f"treated (historical): {list(policy_calendar.treated_states)} | target: {target_state}")

# Defensive, applies in both branches: if the panel happens to include rows for the
# target state at/after its effective week (e.g. you're backtesting against a state
# where the policy already happened), drop them. The baseline forecaster's "last known
# week" must be the last PRE-policy week, exactly like the synthetic simulator gives by
# default -- this exact bug class (forecasting from the wrong starting point) was caught
# during development for the LOSO validation loop; see validation/loso.py::_truncate_panel.
panel = panel[~((panel["state"] == target_state) & (panel["week"] >= target_effective_week))].copy()

categories_to_run = list(CATEGORIES) if USE_SYNTHETIC_DATA else sorted(panel["category"].unique())
covariates = prepare_covariates(state_covariates_raw)

## 3. Exploratory checks

In [ ]:
print("Panel shape:", panel.shape)
display(panel.head())

print("\nStates per category x manufacturer_group:")
display(panel.groupby(["category", "manufacturer_group"])["state"].nunique().rename("n_states"))

print(f"\nTreated (historical) states: {list(policy_calendar.treated_states.keys())}")
print(f"Donor pool ({len(donor_pool_states)} states): {donor_pool_states}")
print(f"Target state: {target_state}  |  target effective week: {target_effective_week}")
print(f"Categories to run: {categories_to_run}")

## 4. Run the causal + forecasting pipeline per category

For each category: Layer 1 (staggered-adoption event study) estimates the pooled
historical effect curve, Layer 3 transports it to the target state via its covariates,
and the baseline forecaster + scenario combiner produce the 13-week Policy vs.
No-Policy scenarios (methodology sections 4-5, 12).


In [ ]:
category_results = {}
for cat in categories_to_run:
    r = run_category_pipeline(
        panel, policy_calendar, covariates,
        target_state=target_state, target_effective_week=target_effective_week,
        donor_states=donor_pool_states, category=cat,
        pre_period_lookback=PRE_PERIOD_LOOKBACK_WEEKS,
        n_bootstrap=N_BOOTSTRAP_MAIN, n_mc=N_MC,
    )
    category_results[cat] = r
    print(f"{cat:15s} transport_scale={r.transport_scale:+.3f}  "
          f"cum %impact mean={r.scenario.cumulative['pct_impact']['mean']:+.1%}")

## 5. Executive business table (methodology section 13.1)

In [ ]:
exec_table = build_executive_table({c: r.scenario for c, r in category_results.items()})
display(exec_table)

## 6. Event study + 13-week scenario chart

In [ ]:
PLOT_CATEGORY = "Vapor" if "Vapor" in category_results else categories_to_run[0]

fig = plot_event_study(category_results[PLOT_CATEGORY].event_study, title=f"{PLOT_CATEGORY} event study")
display(fig)

In [ ]:
fig = plot_scenario_fan_chart(category_results[PLOT_CATEGORY].scenario, PLOT_CATEGORY)
display(fig)

## 7. Cross-category substitution (methodology section 6)

In [ ]:
recon = reconcile_categories({c: r.scenario for c, r in category_results.items()})
display(recon.category_share_of_gross_movement)
print("Total market (reconciled from category sum):", recon.total_market["pct_impact"])

fig = plot_substitution_waterfall(recon.category_share_of_gross_movement)
display(fig)

In [ ]:
# Optional diagnostic -- needs price/promo_depth/distribution_acv populated to detrend
# against; skipped gracefully if your real data doesn't have them.
feat_by_cat = {c: build_aggregated_feature_series(panel, policy_calendar, category=c) for c in categories_to_run}
try:
    corr = category_residual_correlation(feat_by_cat)
    display(corr)
except Exception as e:
    print(f"Skipping residual-correlation diagnostic ({type(e).__name__}: {e}). "
          "This needs price/promo_depth/distribution_acv populated in your panel.")

## 8. Altria vs. competitor decomposition (methodology section 7)

In [ ]:
if {altria_label, competitor_label}.issubset(set(panel["manufacturer_group"].unique())):
    altria_res = run_category_pipeline(
        panel, policy_calendar, covariates, target_state=target_state,
        target_effective_week=target_effective_week, donor_states=donor_pool_states,
        category=PLOT_CATEGORY, manufacturer_group=altria_label,
        pre_period_lookback=PRE_PERIOD_LOOKBACK_WEEKS, n_bootstrap=N_BOOTSTRAP_MAIN, n_mc=N_MC,
    )
    competitor_res = run_category_pipeline(
        panel, policy_calendar, covariates, target_state=target_state,
        target_effective_week=target_effective_week, donor_states=donor_pool_states,
        category=PLOT_CATEGORY, manufacturer_group=competitor_label,
        pre_period_lookback=PRE_PERIOD_LOOKBACK_WEEKS, n_bootstrap=N_BOOTSTRAP_MAIN, n_mc=N_MC,
    )
    decomp = decompose_manufacturer(PLOT_CATEGORY, altria_res.scenario, competitor_res.scenario)

    mfg_table = pd.concat([
        build_executive_table({PLOT_CATEGORY: altria_res.scenario}).assign(View=altria_label),
        build_executive_table({PLOT_CATEGORY: competitor_res.scenario}).assign(View=competitor_label),
    ])
    mfg_table = mfg_table[mfg_table["Metric"] == PLOT_CATEGORY]
    display(mfg_table)
    print(f"Reconciled total (sum of manufacturer groups) %impact mean: "
          f"{decomp.total_from_manufacturer_sum['pct_impact']['mean']:+.1%}  "
          f"(category-level estimate was {category_results[PLOT_CATEGORY].scenario.cumulative['pct_impact']['mean']:+.1%})")

    fig = plot_share_shift(decomp.share_shift, PLOT_CATEGORY)
    display(fig)
else:
    print(f"Skipping Altria/competitor decomposition -- manufacturer_group values "
          f"{sorted(panel['manufacturer_group'].unique())} don't include both "
          f"ALTRIA_LABEL={altria_label!r} and COMPETITOR_LABEL={competitor_label!r}. "
          "Set those in the Configuration cell to match your data.")
    altria_res = competitor_res = decomp = None

## 9. Leave-one-state-out validation (methodology section 10)

Refits the whole stack once per historical treated state, holding it out, and scores
the forecast against that state's *actual* historical post-policy volume. With real
data, only forecast-accuracy metrics (WAPE, interval coverage) are computable -- the
true no-policy counterfactual for a real state is fundamentally unobservable, so the
impact-bias/sign-accuracy/rank-correlation metrics are simulation-only and will show as
`None`/skipped when `USE_SYNTHETIC_DATA = False`.


In [ ]:
if RUN_LOSO_VALIDATION:
    loso_results = run_loso(
        panel, policy_calendar, covariates, donor_pool_states, category=PLOT_CATEGORY,
        n_bootstrap=N_BOOTSTRAP_VALIDATION, n_mc=1500, true_no_policy=true_no_policy_for_loso,
    )
    loso_summary = summarize_loso(loso_results)
    display(loso_summary)
    print("Aggregate LOSO metrics:", loso_aggregate_metrics(loso_summary))
else:
    print("RUN_LOSO_VALIDATION is False -- skipping (set True in the Configuration cell; "
          "expect ~10-15 minutes for a full run across all historical treated states).")
    loso_summary = None

## 10. Placebo-in-time check (methodology section 10.2)

Assigns a fake policy date to genuinely never-treated states and re-runs Layer 1's own
estimator -- the estimated effect should be statistically indistinguishable from zero.
This tests the causal identification itself, independent of any real treated states,
and is fast (no baseline forecaster involved).


In [ ]:
PLACEBO_STATES = donor_pool_states[: min(6, len(donor_pool_states))]
if PLACEBO_STATES:
    placebo = run_placebo_in_time(panel, donor_pool_states, PLACEBO_STATES, category=PLOT_CATEGORY, n_bootstrap=200)
    display(placebo)
    print(f"False positive rate at 90% CI: {placebo['false_positive_at_90pct'].mean():.1%} "
          "(expect roughly ~10% under a well-calibrated null; small-N samples are noisy)")
else:
    print("No donor-pool states available for a placebo check.")
    placebo = None

## 11. Summary & productionizing on Databricks

Fill in the executive table and chart references above into your own summary, e.g.:

> Based on the experience of `{n}` states that have already implemented similar vapor
> policies, we project **{target_state}**'s vapor category volume would be **X% lower**
> over the first 13 weeks post-implementation than it would have been otherwise, with
> roughly **Y%** of that loss shifting into cigarettes/MST rather than leaving the
> tracked nicotine category outright.

**Productionizing this notebook on Databricks** (methodology section 8):

- **Feature store**: land the analysis-ready panel as a Delta table, refreshed on your
  normal scan-data cadence; `vapor_policy_impact.features.engineering` stays the single
  source of truth for the transformation logic either way.
- **Model/version lineage**: log each Layer 1/2/3 fit and baseline-forecaster run to
  **MLflow** (parameters: category, target state, bootstrap/MC sizing; artifacts: the
  executive table, event-study and fan-chart figures) so any business output is
  traceable back to the exact model version that produced it.
- **Retraining cadence**: the causal layer (Layers 1-3) only needs periodic retraining
  (e.g. quarterly, or whenever a new state crosses 13+ weeks post-policy) — schedule it
  as a **Databricks Job**. The baseline forecaster should refresh on every new scan-data
  drop for the target state — a separate, more frequent Job.
- **Parameterized re-runs**: the `dbutils.widgets` wired up in the Configuration section
  let this notebook run as a parameterized Job/Workflow task per target state.
